# AE2 — Reconocimiento del dataset de abandono de clientes (churn)

**Proyecto integrador · Etapa 1: ideación, diseño y documentación de la necesidad**

En este notebook se construye el flujo técnico de la etapa:

1. Configurar la ruta del proyecto para importar las clases reutilizables de `src/clases_po`.
2. Obtener el dataset `scikit-learn/churn-prediction` desde Hugging Face y cargarlo en un `DataFrame`.
3. Hacer una descripción básica: registros, dimensiones, columnas, tipos y valores nulos.
4. Explicar qué representa cada fila y qué variables respaldan la necesidad.
5. Generar automáticamente el resumen en `AE2/resultados/resumen_dataframe.txt` y verificar su contenido.

**Necesidad (escenario hipotético):** el área de retención de clientes de una compañía de telecomunicaciones
necesita conocer qué perfiles de cliente concentran las cancelaciones del servicio, para orientar sus campañas de fidelización.

**Pregunta de análisis:** ¿cómo varía la proporción de clientes que abandonaron el servicio según el tipo de contrato,
la antigüedad, el cargo mensual, el método de pago y el tipo de servicio de internet?

Ejecuta las celdas en orden (**Kernel → Restart & Run All**).

## 1. Preparar el entorno e importar las clases

Igual que en el repositorio del curso, se buscan las carpetas padre hasta encontrar `src/clases_po` y se agrega `src`
a `sys.path`. Así el notebook funciona sin instalar el proyecto como paquete.

In [1]:
import sys
import warnings
from pathlib import Path

# Buscamos la carpeta raíz del proyecto (la que contiene src/clases_po),
# subiendo de carpeta en carpeta desde donde está abierto el notebook.
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent
# Agregamos "src" a las rutas de Python para poder importar nuestras clases.
sys.path.insert(0, str(ROOT / "src"))

# Oculta advertencias informativas de las librerías (no afectan el resultado)
warnings.filterwarnings("ignore")
from huggingface_hub.utils import logging as hf_logging
hf_logging.set_verbosity_error()

import pandas as pd
from clases_po import HFDatasetLoader, Reportes  # nuestras clases de carga y de resumen

# Mostrar todas las columnas al imprimir tablas (el dataset tiene 21).
pd.set_option("display.max_columns", 30)
print("Raíz del proyecto:", ROOT.name)

Raíz del proyecto: PI3-AE2-churn


## 2. Obtener el dataset desde Hugging Face

- **Dataset:** [`scikit-learn/churn-prediction`](https://huggingface.co/datasets/scikit-learn/churn-prediction)
- **Split:** `train` (es el único disponible). Se carga **completo**, sin muestreo.
- **Licencia:** CC-BY-4.0. Es público, así que no requiere token.
- **Procedencia:** según la ficha, son datos de una compañía de telecomunicaciones **ficticia** creados por IBM Sample Datasets.

In [2]:
# Configuramos qué dataset descargar y qué split usar.
loader = HFDatasetLoader(
    dataset_name="scikit-learn/churn-prediction",
    split="train",
)

# Descargamos los datos y los convertimos en un DataFrame de pandas.
df = loader.load()

# Mostramos el origen y el tamaño de lo que se cargó.
loader.info()

Generating train split:   0%|          | 0/7043 [00:00<?, ? examples/s]

Generating train split: 100%|██████████| 7043/7043 [00:00<00:00, 451723.88 examples/s]

{'dataset': 'scikit-learn/churn-prediction',
 'config': None,
 'split': 'train',
 'muestra': 'no (split completo)',
 'filas': 7043,
 'columnas': 21}

## 3. Descripción básica del DataFrame

### 3.1 Algunos registros

In [3]:
# Primeros 5 registros del DataFrame
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
# 5 registros al azar (random_state fija el azar para que siempre salgan los mismos)
df.sample(5, random_state=42)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
185,1024-GUALD,Female,0,Yes,No,1,No,No phone service,DSL,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,24.80,24.8,Yes
2715,0484-JPBRU,Male,0,No,No,41,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Bank transfer (automatic),25.25,996.45,No
3825,3620-EHIMZ,Female,0,Yes,Yes,52,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.35,1031.7,No
1807,6910-HADCM,Female,0,No,No,1,Yes,No,Fiber optic,No,No,Yes,No,No,No,Month-to-month,No,Electronic check,76.35,76.35,Yes
132,8587-XYZSF,Male,0,No,No,67,Yes,No,DSL,No,No,No,Yes,No,No,Two year,No,Bank transfer (automatic),50.55,3260.1,No


### 3.2 Dimensiones

In [5]:
# shape devuelve (cantidad de filas, cantidad de columnas)
filas, columnas = df.shape
print(f"Filas: {filas}")
print(f"Columnas: {columnas}")

Filas: 7043
Columnas: 21


### 3.3 Nombres y tipos de columnas

In [6]:
# Nombre, cantidad de valores no vacíos y tipo de dato de cada columna
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [7]:
# Cuántas columnas hay de cada tipo de dato
df.dtypes.value_counts().rename("cantidad de columnas")

str        18
int64       2
float64     1
Name: cantidad de columnas, dtype: int64

### 3.4 Valores nulos por columna

In [8]:
# isna() marca los valores vacíos y sum() los cuenta por columna
nulos = df.isna().sum().rename("nulos")
nulos.to_frame().T

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
nulos,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


**Observación:** `TotalCharges` (cargos totales) es numérica según su significado, pero viene como texto (`object`).
Algunos registros tienen un espacio en blanco en lugar de un valor. Pandas no los cuenta como nulos, así que se revisan aparte.
Aquí **no se modifica el DataFrame**: solo se identifica el hallazgo para documentarlo como limitación.

In [9]:
# Buscamos los TotalCharges que solo tienen espacios (vacíos "ocultos")
blancos = df["TotalCharges"].str.strip().eq("")
print("Registros con TotalCharges en blanco:", int(blancos.sum()))

# Mostramos esos registros junto con algunas columnas para entenderlos
df.loc[blancos, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

Registros con TotalCharges en blanco: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


## 4. ¿Qué representa cada fila?

Cada fila es **un cliente** de la compañía. Se comprueba que `customerID` no se repite:

In [10]:
# Si customerID nunca se repite, cada fila es un cliente distinto
print("Clientes únicos:", df["customerID"].nunique(), "de", len(df), "filas")
print("¿customerID es único por fila?", df["customerID"].is_unique)

Clientes únicos: 7043 de 7043 filas
¿customerID es único por fila? True


### Variables que respaldan la necesidad

| Variable | Rol en el análisis | Tipo |
|---|---|---|
| `Churn` | Variable central: indica si el cliente abandonó el servicio en el último mes (Yes/No). | Categórica |
| `Contract` | Tipo de contrato (mes a mes, un año, dos años): grupo de comparación principal. | Categórica |
| `tenure` | Antigüedad del cliente en meses: permite comparar por periodos de permanencia. | Numérica |
| `MonthlyCharges` | Cargo mensual: permite comparar niveles de facturación. | Numérica |
| `TotalCharges` | Cargo total acumulado (viene como texto; requiere conversión en la siguiente etapa). | Numérica (como texto) |
| `PaymentMethod` | Método de pago: grupo de comparación. | Categórica |
| `InternetService` | Tipo de servicio de internet (DSL, fibra, ninguno). | Categórica |
| `SeniorCitizen`, `Partner`, `Dependents` | Perfil demográfico básico del cliente. | Categóricas (`SeniorCitizen` codificada 0/1) |

Una primera mirada a las frecuencias de las dos variables principales (solo para reconocer los datos; el análisis
detallado corresponde a las siguientes etapas):

In [11]:
# Cuántos clientes abandonaron (Yes) y cuántos no (No), en cantidad y porcentaje
pd.DataFrame({
    "clientes": df["Churn"].value_counts(),
    "%": (df["Churn"].value_counts(normalize=True) * 100).round(1),
})

,clientes,%
Churn,,
No,5174,73.5
Yes,1869,26.5


In [12]:
# Cuántos clientes hay por tipo de contrato, en cantidad y porcentaje
pd.DataFrame({
    "clientes": df["Contract"].value_counts(),
    "%": (df["Contract"].value_counts(normalize=True) * 100).round(1),
})

,clientes,%
Contract,,
Month-to-month,3875,55.0
Two year,1695,24.1
One year,1473,20.9


In [13]:
# Estadísticas básicas de antigüedad (meses) y cargo mensual
df[["tenure", "MonthlyCharges"]].describe().round(2)

,tenure,MonthlyCharges
count,7043.00,7043.00
mean,32.37,64.76
std,24.56,30.09
min,0.00,18.25
25%,9.00,35.50
50%,29.00,70.35
75%,55.00,89.85
max,72.00,118.75


## 5. Generar automáticamente el resumen `.txt`

`Reportes` recibe el origen de los datos (`loader.info()`) y escribe el resumen del **mismo DataFrame** revisado arriba:
identificador del dataset, split, si es muestra, filas, columnas, tipos y nulos por columna.
La carpeta `AE2/resultados/` se crea si no existe y el archivo se regenera en cada ejecución (UTF-8).

In [14]:
# Creamos el generador de reportes y le pasamos el origen de los datos
reportes = Reportes(origen=loader.info())

# Guardamos el resumen del MISMO DataFrame en AE2/resultados (se sobrescribe en cada ejecución)
ruta_resumen = reportes.guardar_reporte(df, ROOT / "AE2" / "resultados" / "resumen_dataframe.txt")
print("Resumen guardado en:", ruta_resumen.relative_to(ROOT).as_posix())

Resumen guardado en: AE2/resultados/resumen_dataframe.txt


### 5.1 Verificar el contenido del archivo generado

In [15]:
# Leemos el archivo recién creado y lo mostramos para verificarlo
contenido = ruta_resumen.read_text(encoding="utf-8")
print(contenido)

RESUMEN DEL DATAFRAME
Generado: 2026-10-25 20:03:06

Origen de los datos:
- Dataset (Hugging Face): scikit-learn/churn-prediction
- Configuración: por defecto
- Split: train
- Muestra: no (split completo)

Dimensiones:
- Filas: 7043
- Columnas: 21
- Filas duplicadas: 0

Columnas, tipos y valores nulos:
                     tipo  nulos  valores_unicos
columna                                         
customerID            str      0            7043
gender                str      0               2
SeniorCitizen       int64      0               2
Partner               str      0               2
Dependents            str      0               2
tenure              int64      0              73
PhoneService          str      0               2
MultipleLines         str      0               3
InternetService       str      0               3
OnlineSecurity        str      0               3
OnlineBackup          str      0               3
DeviceProtection      str      0               3
TechSuppor

In [16]:
# Comprobaciones: el archivo corresponde al DataFrame revisado en este notebook.
# Si alguna falla, assert detiene el notebook con un error.
assert "scikit-learn/churn-prediction" in contenido
assert f"- Filas: {len(df)}" in contenido
assert f"- Columnas: {len(df.columns)}" in contenido
assert all(columna in contenido for columna in df.columns)
print("✔ El resumen .txt coincide con el DataFrame explorado:", df.shape)

✔ El resumen .txt coincide con el DataFrame explorado: (7043, 21)


## 6. Conclusión de la etapa

- El dataset tiene **7 043 clientes (filas)** y **21 variables (columnas)**. Cada fila es un cliente.
- No hay valores nulos reconocidos por pandas, pero `TotalCharges` viene como texto y tiene **11 registros en blanco**.
  Corresponden a clientes con antigüedad 0, y eso se documenta como limitación.
- Las variables `Churn`, `Contract`, `tenure`, `MonthlyCharges`, `PaymentMethod` e `InternetService` permiten responder la
  pregunta de análisis comparando la proporción de abandono entre grupos de clientes.
- Los datos son de una compañía **ficticia** (IBM Sample Datasets). Las conclusiones describen este conjunto de datos,
  no a una empresa real.